In [66]:
# Set project root so "from src..." works. Edit the path below if you get ModuleNotFoundError.
import sys
from pathlib import Path
PROJECT_ROOT = Path("/Users/tomosborne/PycharmProjects/PowerPlan")  # <-- edit if your repo is elsewhere
if PROJECT_ROOT.exists():
    sys.path.insert(0, str(PROJECT_ROOT))

# Energy balancing model — Usage

**Inputs:** latitude, longitude (location), solar/wind **capacity** (kW), and **generation-type** params from `src.data.energy_tiers` (budget, mid, premium).

**Flows:**
1. **Daily generation** — `get_flux_daily` + `get_generation`: short-range forecast → daily solar/wind kWh.
2. **Last year monthly** — `get_flux_monthly_last_year`: Historical API → 12 months GHI + wind (used for optimisation).
3. **Optimisation** — `get_optimised_system(..., flux_source='last_year_monthly')`: sizes and prices system using last year's weather; returns **financials over 0, 5 and 10 years**. Use `flux_source='forecast'` for 7-day forecast instead.

In [67]:
# Location + capacity + generation types (run project-root cell first)
import importlib
import sys
# Force reload from disk so the kernel sees latest code (fixes stale import errors)
for name in list(sys.modules):
    if name in ("src.models.energy_balancing", "src.api.get_weather") or name.startswith("src.models.energy_balancing."):
        del sys.modules[name]
import src.models.energy_balancing as _eb
importlib.reload(_eb)
get_generation = _eb.get_generation
get_flux_daily = _eb.get_flux_daily
get_flux_monthly_last_year = _eb.get_flux_monthly_last_year
get_optimised_system = _eb.get_optimised_system
DEFAULT_PRICING = _eb.DEFAULT_PRICING
from src.data.energy_tiers import SOLAR_TIERS, WIND_TIERS

# Inputs: location and capacity
latitude = 51.4552   # e.g. Bristol, UK
longitude = -2.5966
solar_capacity_kw = 4.0
wind_capacity_kw = 2.0

# Generation types (stored in src.data.energy_tiers)
solar_type = SOLAR_TIERS["budget"]
wind_type = WIND_TIERS["budget"]

# Optional: date range (default: next 7 days from today)
# start_date, end_date = "2026-02-20", "2026-02-26"

daily = get_generation(
    latitude, longitude,
    solar_capacity_kw, wind_capacity_kw,
    solar_type, wind_type,
)
daily

,ghi_mj_per_m2,wind_speed_10m_max,solar_gen_kwh,wind_gen_kwh,total_gen_kwh
date,,,,,
2026-03-04 00:00:00+00:00,9.86,3.9,8.444542,0.480000,8.924542
2026-03-05 00:00:00+00:00,11.02,4.2,9.438018,0.853333,10.291351
2026-03-06 00:00:00+00:00,NaN,3.8,0.000000,0.379259,0.379259
2026-03-07 00:00:00+00:00,NaN,3.8,0.000000,0.379259,0.379259
2026-03-08 00:00:00+00:00,NaN,3.3,0.000000,0.053333,0.053333
2026-03-09 00:00:00+00:00,NaN,3.0,0.000000,0.000000,0.000000
2026-03-10 00:00:00+00:00,NaN,3.2,0.000000,0.023704,0.023704


In [68]:
# Summary for the period: total solar, wind, and combined (kWh)
print("Period total — solar:", round(daily["solar_gen_kwh"].sum(), 1), "kWh, wind:", round(daily["wind_gen_kwh"].sum(), 1), "kWh, total:", round(daily["total_gen_kwh"].sum(), 1), "kWh")

Period total — solar: 17.9 kWh, wind: 2.2 kWh, total: 20.1 kWh


In [69]:
# Last year's weather by month (Historical API) — used for optimisation below
try:
    flux_monthly = get_flux_monthly_last_year(latitude, longitude)
    print("Last year monthly flux (GHI = shortwave sum MJ/m², wind = mean max m/s):")
except Exception as e:
    print("Could not load last-year monthly data (Historical API):", e)
    print("Optimisation cell will fall back to flux_source='forecast'.")
    flux_monthly = None
flux_monthly

Last year monthly flux (GHI = shortwave sum MJ/m², wind = mean max m/s):


,ghi_mj_per_m2,wind_speed_10m_max,days_in_month
month,,,
1,100.430000,5.867279,31
2,133.580002,6.174774,28
3,367.750000,4.611729,31
4,520.820007,5.177849,30
5,660.460022,5.332867,31
6,648.210022,5.992554,30
7,654.010010,5.250819,31
8,537.010010,5.361267,31
9,366.770020,5.985127,30


In [70]:
# Optimisation: size and price (last year's monthly weather, or forecast if that fails)
annual_consumption_kwh = 3500  # your annual demand (kWh)

try:
    result = get_optimised_system(
        latitude, longitude,
        annual_consumption_kwh,
        solar_type, wind_type,
        solar_max_kw=20.0,
        wind_max_kw=10.0,
        step_kw=0.5,
        optimize_over_years=5.0,
        flux_source="last_year_monthly",
        min_wind_kw=0.5,   # require at least 0.5 kW wind (set to 0 to allow wind=0)
    )
except Exception as e:
    print("Last-year monthly failed, using 7-day forecast instead:", e)
    result = get_optimised_system(
        latitude, longitude, annual_consumption_kwh, solar_type, wind_type,
        solar_max_kw=20.0, wind_max_kw=10.0, step_kw=0.5, optimize_over_years=5.0,
        flux_source="forecast",
        min_wind_kw=0.5,
    )

print("--- Optimal system (sized to minimise cost over 5 years) ---")
print("Flux data:", result["flux_source"], "(period:", result["flux_period_days"], "days)")
print("Capacity: solar", result["optimal_solar_kw"], "kW  |  wind", result["optimal_wind_kw"], "kW  (total", result["total_capacity_kw"], "kW)")
print("Annual demand:", result["annual_demand_kwh"], "kWh  |  Annual generation:", result["annual_generation_kwh"], "kWh")
print("  → solar:", result.get("annual_solar_generation_kwh", 0), "kWh  |  wind:", result.get("annual_wind_generation_kwh", 0), "kWh")
print("Demand met from own generation:", result["demand_met_from_generation_pct"], "%")
print()
print("Capex: solar £" + str(result.get("solar_capex", 0)) + "  |  wind £" + str(result.get("wind_capex", 0)))
print("Payback: solar", result.get("payback_solar_years") or "—", "years  |  wind", result.get("payback_wind_years") or "—", "years")
print()
print("Financials:")
print("  0 year (upfront):  £", result["financials_0_year"]["total"], " (capex only)")
print("  5 year total:      £", result["financials_5_year"]["total"], " (capex + 5 yr opex)")
print("  10 year total:    £", result["financials_10_year"]["total"], " (capex + 10 yr opex)")
print("  (opex = grid import cost − export revenue)")
result

--- Optimal system (sized to minimise cost over 5 years) ---
Flux data: last_year_monthly (period: 365 days)
Capacity: solar 1.5 kW  |  wind 0.5 kW  (total 2.0 kW)
Annual demand: 3500 kWh  |  Annual generation: 1839.3 kWh
  → solar: 1405.9 kWh  |  wind: 433.4 kWh
Demand met from own generation: 52.6 %

Capex: solar £2250.0  |  wind £1250.0
Payback: solar 6.4 years  |  wind 11.5 years

Financials:
  0 year (upfront):  £ 3500.0  (capex only)
  5 year total:      £ 5575.83  (capex + 5 yr opex)
  10 year total:    £ 7651.66  (capex + 10 yr opex)
  (opex = grid import cost − export revenue)


{'optimal_solar_kw': 1.5,
 'optimal_wind_kw': 0.5,
 'total_capacity_kw': 2.0,
 'annual_demand_kwh': 3500,
 'annual_generation_kwh': 1839.3,
 'annual_solar_generation_kwh': 1405.9,
 'annual_wind_generation_kwh': 433.4,
 'annual_import_kwh': 1660.7,
 'annual_export_kwh': 0.0,
 'demand_met_from_generation_pct': 52.6,
 'capex': 3500.0,
 'solar_capex': 2250.0,
 'wind_capex': 1250.0,
 'annual_net_opex': 415.17,
 'payback_solar_years': 6.4,
 'payback_wind_years': 11.5,
 'financials_0_year': {'capex': 3500.0, 'opex_total': 0.0, 'total': 3500.0},
 'financials_5_year': {'capex': 3500.0,
  'opex_total': 2075.83,
  'total': 5575.83},
 'financials_10_year': {'capex': 3500.0,
  'opex_total': 4151.66,
  'total': 7651.66},
 'flux_source': 'last_year_monthly',
 'flux_period_days': 365}

In [ ]:
# Optimal plan over the year: solar vs wind generation by month (demand = annual/12 per month)
if result.get("monthly_balance") is not None:
    print("Monthly generation and balance (solar + wind over the year):")
    result["monthly_balance"]
else:
    print("Monthly breakdown available only when using flux_source='last_year_monthly'.")

In [71]:
# Optional: daily flux from forecast (no generation) — for comparison with monthly historical
flux_daily = get_flux_daily(latitude, longitude)
flux_daily

,ghi_mj_per_m2,wind_speed_10m_max
date,,
2026-03-04 00:00:00+00:00,9.86,3.9
2026-03-05 00:00:00+00:00,11.02,4.2
2026-03-06 00:00:00+00:00,NaN,3.8
2026-03-07 00:00:00+00:00,NaN,3.8
2026-03-08 00:00:00+00:00,NaN,3.3
2026-03-09 00:00:00+00:00,NaN,3.0
2026-03-10 00:00:00+00:00,NaN,3.2


In [72]:
# Switch generation types: SOLAR_TIERS["mid"], WIND_TIERS["premium"], etc.
# get_generation(..., SOLAR_TIERS["mid"], WIND_TIERS["premium"])
# get_optimised_system(..., flux_source="forecast")  # use 7-day forecast instead of last year